# GENESIS 12411-0013 hero dataset
Reproducible profiling and Pandas–DuckDB reconciliation. Population is resident population, not GKV coverage or clinical eligibility.

In [ ]:
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from final_project_stackfuel.hero_pipeline import ROOT
processed_path = ROOT / '02_processed_data/hero/population_state_age_sex.parquet'
database_path = ROOT / '02_processed_data/hero/population_hero.duckdb'
population = pd.read_parquet(processed_path)
population.shape

## Quality and missingness
Null age values occur only on official total rows and are structural, not unknown ages.

In [ ]:
quality = pd.DataFrame({'dtype': population.dtypes.astype(str), 'missing': population.isna().sum(), 'cardinality': population.nunique(dropna=False)})
quality

## Pandas transformation
Create a 2025 state ranking from official total rows. This does not distribute national values regionally.

In [ ]:
state_2025 = (population.query("year == 2025 and is_age_total and is_sex_total")
              .loc[:, ['state_code','state_name','population_persons']]
              .sort_values('population_persons', ascending=False)
              .assign(population_rank=lambda x: x.population_persons.rank(method='dense', ascending=False).astype(int)))
state_2025.head()

## DuckDB and reconciliation

In [ ]:
with duckdb.connect(str(database_path), read_only=True) as con:
    sql_2025 = con.execute("SELECT state_code, population_persons, population_rank FROM population_state_summary_sql WHERE year=2025 ORDER BY population_rank").df()
check = state_2025.merge(sql_2025, on='state_code', suffixes=('_pandas','_sql'), validate='one_to_one')
assert (check.population_persons_pandas == check.population_persons_sql).all()
assert (check.population_rank_pandas == check.population_rank_sql).all()
check.head()

## Distribution

In [ ]:
ax = state_2025.plot.barh(x='state_name', y='population_persons', figsize=(8,6), legend=False, title='Resident population by Bundesland, 2025')
ax.set_xlabel('Persons')
ax.invert_yaxis()
plt.tight_layout()

## Conclusions and limitations
The dataset supports genuine state, sex and age analysis. Results through 2021 use the Census 2011 basis; 2022 onward uses Census 2022, so the 2021–2022 boundary is not a continuous trend. Resident population must not be labelled as GKV-insured, clinically eligible or treatable population.